In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install wandb torch torchvision pandas numpy matplotlib seaborn scikit-learn mlflow dagshub
!pip install kaggle neuralforecast

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212

In [ ]:
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c walmart-recruiting-store-sales-forecasting
!unzip -q walmart-recruiting-store-sales-forecasting.zip

100% 2.70M/2.70M [00:00<00:00, 229MB/s]



In [ ]:
!unzip -q train.csv.zip
!unzip -q stores.csv.zip
!unzip -q test.csv.zip
!unzip -q features.csv.zip

unzip:  cannot find or open stores.csv.zip, stores.csv.zip.zip or stores.csv.zip.ZIP.


In [ ]:
import dagshub
import mlflow

dagshub.init(repo_owner='tsarc21', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=908b6672-cad7-4609-b67c-e11aa9af5e34&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=ee39d90e9ee91cb4440fde3271eca4140ab05ecc0da55523c394be4d272b5972




Accessing as tsarc21

Initialized MLflow to track repo "tsarc21/Walmart-Recruiting---Store-Sales-Forecasting"

Repository tsarc21/Walmart-Recruiting---Store-Sales-Forecasting initialized!

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, TiDE
from utilsforecast.preprocessing import fill_gaps



train_df = train_nf[["unique_id", "ds", "y", "IsHoliday"]].copy()
val_df = val_nf[["unique_id", "ds", "y", "IsHoliday"]].copy()

train_df["ds"] = pd.to_datetime(train_df["ds"])
val_df["ds"] = pd.to_datetime(val_df["ds"])
global_train_max = train_df["ds"].max()
series_max_dates = train_df.groupby("unique_id")["ds"].max()
aligned_ids = series_max_dates[series_max_dates == global_train_max].index

train_df = train_df[train_df["unique_id"].isin(aligned_ids)].reset_index(drop=True)
val_df = val_df[val_df["unique_id"].isin(aligned_ids)].reset_index(drop=True)

train_df = fill_gaps(df=train_df, freq="W-FRI")
train_df["IsHoliday"] = train_df["IsHoliday"].fillna(0)
train_df["y"] = train_df["y"].ffill().bfill()

max_available_len = train_df.groupby("unique_id").size().min()
input_size = min(52, max_available_len)

series_length = train_df.groupby("unique_id").size()
valid_ids = series_length[series_length >= input_size].index

train_df = train_df[train_df["unique_id"].isin(valid_ids)].reset_index(drop=True)
val_df = val_df[val_df["unique_id"].isin(valid_ids)].reset_index(drop=True)

print(f"Retained Aligned Series: {train_df['unique_id'].nunique()}")
print(f"Selected Input Lookback Horizon: {input_size} weeks")



horizon = 40

nhits_model = NHITS(
    h=horizon,
    input_size=input_size,
    n_blocks=[2, 2, 2],
    mlp_units=[[256, 256], [256, 256], [256, 256]],
    learning_rate=0.0003,
    batch_size=32,
    max_steps=500,
    futr_exog_list=["IsHoliday"],
    val_check_steps=20,
    early_stop_patience_steps=5,
    random_seed=42
)

tide_model = TiDE(
    h=horizon,
    input_size=input_size,
    decoder_output_dim=16,
    hidden_size=128,
    num_encoder_layers=2,
    num_decoder_layers=2,
    learning_rate=0.0005,
    batch_size=32,
    max_steps=500,
    futr_exog_list=["IsHoliday"],
    val_check_steps=20,
    early_stop_patience_steps=5,
    random_seed=42
)

nf = NeuralForecast(
    models=[nhits_model, tide_model],
    freq="W-FRI"
)

nf.fit(
    df=train_df[["unique_id", "ds", "y", "IsHoliday"]],
    val_df=val_df[["unique_id", "ds", "y", "IsHoliday"]]
)


futr_df = nf.make_future_dataframe()
futr_df = futr_df.merge(
    val_df[["unique_id", "ds", "IsHoliday"]],
    on=["unique_id", "ds"],
    how="left"
).fillna({"IsHoliday": 0})

predictions = nf.predict(futr_df=futr_df).reset_index()

predictions["Ensemble"] = 0.5 * predictions["NHITS"] + 0.5 * predictions["TiDE"]

val_result = val_df.merge(predictions, on=["unique_id", "ds"], how="inner")

def compute_wmae(df, pred_col):
    weights = np.where(df["IsHoliday"] == 1, 5, 1)
    return np.sum(weights * np.abs(df["y"] - df[pred_col])) / np.sum(weights)

print("\n" + "=" * 50)
print(f"NHITS WMAE:    {compute_wmae(val_result, 'NHITS'):.4f}")
print(f"TiDE WMAE:     {compute_wmae(val_result, 'TiDE'):.4f}")
print(f"ENSEMBLE WMAE: {compute_wmae(val_result, 'Ensemble'):.4f}")
print("=" * 50)

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:lightning_fabric.utilities.seed:Seed set to 42


Retained Aligned Series: 2739
Selected Input Lookback Horizon: 52 weeks


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:124: UserWarning: Initializing zero-element tensors is a no-op
  init.kaiming_uniform_(self.weight, a=math.sqrt(5))
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler              | TemporalNorm  | 0      | train
6 | blocks              | ModuleList    | 1.4 M  | train
-

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Predicting: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]


NHITS WMAE:    nan
TiDE WMAE:     nan
ENSEMBLE WMAE: nan


/tmp/ipykernel_517/3210856330.py:120: RuntimeWarning: invalid value encountered in scalar divide
  return np.sum(weights * np.abs(df["y"] - df[pred_col])) / np.sum(weights)


In [ ]:
import numpy as np
import pandas as pd
import torch
import mlflow

from sklearn.metrics import mean_absolute_error
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
from neuralforecast.losses.pytorch import MAE
from utilsforecast.preprocessing import fill_gaps



class WeightedMAELoss(MAE):
    def __init__(self):
        super().__init__()

    def forward(self, y_hat, y, mask=None, futr_exog=None):
        err = torch.abs(y_hat - y)
        if futr_exog is not None and futr_exog.shape[-1] > 0:
            is_holiday = futr_exog[..., 0:1]
            weights = torch.where(is_holiday > 0.5, 5.0, 1.0)
        else:
            weights = 1.0

        weighted_err = err * weights
        if mask is not None:
            weighted_err = weighted_err * mask

        return torch.mean(weighted_err)


def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday == 1, 5.0, 1.0)
    abs_errors = np.abs(y_true - y_pred)
    return np.sum(weights * abs_errors) / np.sum(weights)



train_df = train_nf[["unique_id", "ds", "y", "IsHoliday"]].copy()
val_df = val_nf[["unique_id", "ds", "y", "IsHoliday"]].copy()

train_df["ds"] = pd.to_datetime(train_df["ds"])
val_df["ds"] = pd.to_datetime(val_df["ds"])

global_train_max = train_df["ds"].max()
series_max_dates = train_df.groupby("unique_id")["ds"].max()
aligned_ids = series_max_dates[series_max_dates == global_train_max].index

train_df = train_df[train_df["unique_id"].isin(aligned_ids)].reset_index(drop=True)
val_df = val_df[val_df["unique_id"].isin(aligned_ids)].reset_index(drop=True)

train_df = fill_gaps(df=train_df, freq="W-FRI")
train_df["IsHoliday"] = train_df["IsHoliday"].fillna(0).astype(int)
val_df["IsHoliday"] = val_df["IsHoliday"].fillna(0).astype(int)
train_df["y"] = train_df["y"].ffill().bfill()

holiday_stats = train_df.groupby(["unique_id", "IsHoliday"])["y"].mean().unstack(fill_value=0)

non_holiday_mean = holiday_stats[0] if 0 in holiday_stats.columns else pd.Series(1.0, index=holiday_stats.index)
holiday_mean = holiday_stats[1] if 1 in holiday_stats.columns else pd.Series(1.0, index=holiday_stats.index)

lift_series = holiday_mean / np.maximum(non_holiday_mean, 1e-5)
lift_series = lift_series.replace([np.inf, np.nan], 1.4).clip(lower=1.1, upper=2.5)
holiday_lift = lift_series.to_dict()

train_df["lift"] = train_df["unique_id"].map(holiday_lift).fillna(1.4)
train_df["y_base"] = np.where(
    train_df["IsHoliday"] == 1,
    train_df["y"] / train_df["lift"],
    train_df["y"]
)

nf_train_df = train_df[["unique_id", "ds", "IsHoliday"]].copy()
nf_train_df["y"] = train_df["y_base"]



mlflow.set_experiment("PatchTST_Target_Normalized")

with mlflow.start_run(run_name="PatchTST_Sub3000_Fixed_Pipeline"):

    model = PatchTST(
        h=40,
        input_size=52,
        patch_len=8,
        stride=4,
        hidden_size=256,
        n_heads=8,
        encoder_layers=3,
        dropout=0.1,
        learning_rate=0.0005,
        batch_size=32,
        max_steps=600,
        loss=WeightedMAELoss(),
        revin=True,
        start_padding_enabled=True,
        random_seed=42
    )

    nf = NeuralForecast(models=[model], freq="W-FRI")

    nf.fit(df=nf_train_df)

    val_pred = (
        nf.predict()
        .reset_index()
        .rename(columns={"PatchTST": "raw_pred"})
    )

    val_result = val_df.merge(val_pred, on=["unique_id", "ds"], how="inner")
    val_result["lift"] = val_result["unique_id"].map(holiday_lift).fillna(1.4)

    val_result["prediction"] = np.where(
        val_result["IsHoliday"] == 1,
        val_result["raw_pred"] * val_result["lift"],
        val_result["raw_pred"]
    )
    val_result = val_result.dropna(subset=["y", "prediction"])

    val_wmae_score = wmae(
        val_result["y"].values,
        val_result["prediction"].values,
        val_result["IsHoliday"].values
    )
    val_mae_score = mean_absolute_error(val_result["y"], val_result["prediction"])

    cv_train_df = nf.cross_validation(
        df=nf_train_df,
        val_size=40,
        n_windows=1
    ).reset_index()

    cv_train_df = cv_train_df.merge(
        train_df[["unique_id", "ds", "IsHoliday"]],
        on=["unique_id", "ds"],
        how="left"
    )
    cv_train_df["IsHoliday"] = cv_train_df["IsHoliday"].fillna(0).astype(int)

    cv_train_df["lift"] = cv_train_df["unique_id"].map(holiday_lift).fillna(1.4)

    cv_train_df["y_true_actual"] = np.where(
        cv_train_df["IsHoliday"] == 1,
        cv_train_df["y"] * cv_train_df["lift"],
        cv_train_df["y"]
    )
    cv_train_df["y_pred_actual"] = np.where(
        cv_train_df["IsHoliday"] == 1,
        cv_train_df["PatchTST"] * cv_train_df["lift"],
        cv_train_df["PatchTST"]
    )

    train_wmae_score = wmae(
        cv_train_df["y_true_actual"].values,
        cv_train_df["y_pred_actual"].values,
        cv_train_df["IsHoliday"].values
    )

    print("=" * 70)
    print(f"EXACT TRAIN WMAE:      {train_wmae_score:.4f}")
    print(f"VALIDATION WMAE:       {val_wmae_score:.4f}")
    print(f"VALIDATION MAE:        {val_mae_score:.4f}")
    print("=" * 70)

    mlflow.log_metric("train_wmae", float(train_wmae_score))
    mlflow.log_metric("validation_wmae", float(val_wmae_score))
    mlflow.log_metric("validation_mae", float(val_mae_score))

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | WeightedMAELoss   | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 1.3 M  | train
------------------------------------------------------------------
1.3 M     Trainable params
3 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=600` reached.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

EXACT TRAIN WMAE:      3148.2363
VALIDATION WMAE:       1799.2067
VALIDATION MAE:        1750.3603
🏃 View run PatchTST_Sub3000_Fixed_Pipeline at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/23/runs/93d423062dd844a5b742b04e41580051
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/23


In [ ]:
import numpy as np
import pandas as pd
import torch

from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
from neuralforecast.losses.pytorch import MAE
from utilsforecast.preprocessing import fill_gaps



test_raw = pd.read_csv("test.csv")

test_df = test_raw.copy()
test_df["ds"] = pd.to_datetime(test_df["Date"])
test_df["unique_id"] = test_df["Store"].astype(str) + "_" + test_df["Dept"].astype(str)
test_df["IsHoliday"] = test_df["IsHoliday"].astype(int)
test_df = test_df[["unique_id", "ds", "IsHoliday"]].copy()

full_train_df = pd.concat([
    train_nf[["unique_id", "ds", "y", "IsHoliday"]],
    val_nf[["unique_id", "ds", "y", "IsHoliday"]]
], ignore_index=True)

full_train_df["ds"] = pd.to_datetime(full_train_df["ds"])

global_train_max = full_train_df["ds"].max()
series_max_dates = full_train_df.groupby("unique_id")["ds"].max()
aligned_ids = series_max_dates[series_max_dates == global_train_max].index

full_train_df = full_train_df[full_train_df["unique_id"].isin(aligned_ids)].reset_index(drop=True)

full_train_df = fill_gaps(df=full_train_df, freq="W-FRI")
full_train_df["IsHoliday"] = full_train_df["IsHoliday"].fillna(0).astype(int)
full_train_df["y"] = full_train_df["y"].ffill().bfill()

holiday_stats = full_train_df.groupby(["unique_id", "IsHoliday"])["y"].mean().unstack(fill_value=0)

non_holiday_mean = holiday_stats[0] if 0 in holiday_stats.columns else pd.Series(1.0, index=holiday_stats.index)
holiday_mean = holiday_stats[1] if 1 in holiday_stats.columns else pd.Series(1.0, index=holiday_stats.index)

lift_series = holiday_mean / np.maximum(non_holiday_mean, 1e-5)
lift_series = lift_series.replace([np.inf, np.nan], 1.4).clip(lower=1.1, upper=2.5)
holiday_lift = lift_series.to_dict()

full_train_df["lift"] = full_train_df["unique_id"].map(holiday_lift).fillna(1.4)
full_train_df["y_base"] = np.where(
    full_train_df["IsHoliday"] == 1,
    full_train_df["y"] / full_train_df["lift"],
    full_train_df["y"]
)

nf_full_train = full_train_df[["unique_id", "ds", "IsHoliday"]].copy()
nf_full_train["y"] = full_train_df["y_base"]



forecast_horizon = test_df["ds"].nunique()

final_model = PatchTST(
    h=forecast_horizon,
    input_size=52,
    patch_len=8,
    stride=4,
    hidden_size=256,
    n_heads=8,
    encoder_layers=3,
    dropout=0.1,
    learning_rate=0.0005,
    batch_size=32,
    max_steps=600,
    loss=MAE(),
    revin=True,
    start_padding_enabled=True,
    random_seed=42
)

nf_final = NeuralForecast(models=[final_model], freq="W-FRI")
nf_final.fit(df=nf_full_train)

raw_test_preds = (
    nf_final.predict()
    .reset_index()
    .rename(columns={"PatchTST": "raw_pred"})
)

test_result = test_df.merge(raw_test_preds, on=["unique_id", "ds"], how="left")
test_result["lift"] = test_result["unique_id"].map(holiday_lift).fillna(1.4)

test_result["Weekly_Sales"] = np.where(
    test_result["IsHoliday"] == 1,
    test_result["raw_pred"] * test_result["lift"],
    test_result["raw_pred"]
)

submission_df = test_result[["unique_id", "ds", "Weekly_Sales"]].copy()
submission_df["Weekly_Sales"] = pd.to_numeric(submission_df["Weekly_Sales"], errors="coerce").fillna(0.0)

submission_df["Id"] = submission_df["unique_id"].astype(str) + "_" + submission_df["ds"].dt.strftime("%Y-%m-%d")
submission_df = submission_df[["Id", "Weekly_Sales"]].sort_values("Id").reset_index(drop=True)

submission_df.to_csv("submission.csv", index=False)

print("=" * 70)
print("SUCCESS: Final test predictions saved to 'submission.csv'")
print(f"Total Prediction Rows: {len(submission_df)}")
print(submission_df.head(10))
print("=" * 70)

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 1.3 M  | train
------------------------------------------------------------------
1.3 M     Trainable params
3 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=600` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

SUCCESS: Final test predictions saved to 'submission.csv'
Total Prediction Rows: 115064
                 Id  Weekly_Sales
0  10_10_2012-11-02  43708.601562
1  10_10_2012-11-09  46941.164062
2  10_10_2012-11-16  44705.250000
3  10_10_2012-11-23  50114.405859
4  10_10_2012-11-30  45747.007812
5  10_10_2012-12-07  50115.382812
6  10_10_2012-12-14  51038.800781
7  10_10_2012-12-21  56508.398438
8  10_10_2012-12-28  48315.708203
9  10_10_2013-01-04  41835.863281


In [ ]:
!kaggle competitions submit -c walmart-recruiting-store-sales-forecasting -f submission.csv  -m "Message"

100% 3.53M/3.53M [00:00<00:00, 5.09MB/s]
Successfully submitted to Walmart Recruiting - Store Sales Forecasting

In [ ]:
import mlflow
import os

model_path = "./best_patchtst_model_4"
best_model.save(path=model_path)

with mlflow.start_run(run_name="Best_PatchTST_Artifact", nested=True):
    mlflow.log_artifacts(model_path, artifact_path="neuralforecast_model")
    mlflow.log_params(best_params)
    mlflow.log_metric("best_validation_wmae", float(best_wmae))

print("Model successfully logged to MLflow as an artifact!")

🏃 View run Best_PatchTST_Artifact at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/23/runs/cc59ec3214dc4e99ba3b47de9121905a
🧪 View experiment at: https://dagshub.com/tsarc21/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/23
Model successfully logged to MLflow as an artifact!
